In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
races_df = spark.read.parquet(f"{processed_folder_path}/races*")
races_df.printSchema()

In [0]:
circuits_df = spark.read.parquet(f"{processed_folder_path}/circuits")
circuits_df.printSchema()

In [0]:
drivers_df = spark.read.parquet(f"{processed_folder_path}/drivers")
drivers_df.printSchema()

In [0]:
constructor_df = spark.read.parquet(f"{processed_folder_path}/constructors")
constructor_df.printSchema()

In [0]:
results_df = spark.read.parquet(f"{processed_folder_path}/results*")
results_df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp
race_results_df = races_df \
    .join(circuits_df, circuits_df.circuit_id == races_df.circuit_id, "inner") \
    .join(results_df, results_df.race_id == races_df.race_id, "inner") \
    .join(drivers_df, drivers_df.driver_id == results_df.driver_id, "inner") \
    .join(constructor_df, constructor_df.constructor_id == results_df.constructor_id, "inner") \
    .select(
        races_df.race_year,
        races_df.name.alias("race_name"),
        races_df.race_timestamp.alias("race_date"),
        circuits_df.location,
        drivers_df.name.alias("driver_name"),
        drivers_df.number.alias("driver_number"),
        drivers_df.nationality.alias("driver_nationality"),
        constructor_df.name.alias("team"),
        results_df.grid,
        results_df.fastest_lap,
        results_df.time.alias("race_time"),
        results_df.points,
        results_df.position
    ).withColumn("created_date", current_timestamp())
display(race_results_df.filter("race_year == 2020 and race_name == 'Abu Dhabi Grand Prix'").orderBy(race_results_df.points.desc()))

In [0]:
race_results_df.write.mode("overwrite").partitionBy("race_year").parquet(f"{presentation_folder_path}/race_results")